# Non-Linear Economy Estimation (ANN)

Please note that this notebook uses a venv which points to a base python version of **3.13**, some functionality may be limited if using an older version of python.

## All Imports

- Vanilla Modules:
    - typing (optional)
        - just for static type checking and code hygeine, not functionally required.
    - dataclasses (optional)
        - class type of choice for SVAR return object, also not necessary but functionally optimal.
    - datetime
        - Used for datetime.now() to set the end date bound for out of sample data.
    
- 3rd party:
    - fedfred
        - This is the library which I wrote, published, and currently maintain under the MIT License
        - Makes API requests to the FRED database and returns strongly typed and structured objects as well as dataframes to reduce boilerplate code.
    - pandas
        - dataframe backend of choice, if computational speed or parallalelization becomes a necessity the fedfred backend accomodates dask and polars.
    - numpy
        - for mathematical ops and pandas interop
    - statsmodels
        - for OLS and potentially VAR model if necessary.
    - matplotlib
        - Used for plotting results.
    - 

In [48]:
%pip install scikit-learn seaborn torch torchvision gymnasium stable-baselines3 --quiet

Note: you may need to restart the kernel to use updated packages.


In [49]:
import fedfred as fed
import pandas as pd
import numpy as np
from dataclasses import dataclass
import statsmodels.api as sm
import datetime
import matplotlib.pyplot as plt

In [50]:
%matplotlib widget

In [51]:
%%capture
%run -i ./notebooks/linear_environment.ipynb # Import the linear environment

## Data Check

In [52]:
# Series Check
print(pi.head())
print(pi.tail())
print(y_gap.head())
print(y_gap.tail())
print(i.head())
print(i.tail())

# Frame Check
print(df.head())
print(df.tail())
print(df.index.min(), "→", df.index.max())

date
1987-07-01    2.66122
1987-10-01    2.91876
1988-01-01    3.06577
1988-04-01    3.35274
1988-07-01    3.80207
Name: pi, dtype: float64
date
2006-04-01    3.35642
2006-07-01    3.13805
2006-10-01    2.66316
2007-01-01    2.91019
2007-04-01    2.72883
Name: pi, dtype: float64
date
1987-07-01   -0.388640
1987-10-01    0.526730
1988-01-01    0.260731
1988-04-01    0.788853
1988-07-01    0.595530
Name: y, dtype: float64
date
2006-04-01    1.319357
2006-07-01    0.978513
2006-10-01    1.380249
2007-01-01    1.210046
2007-04-01    1.336951
Name: y, dtype: float64
date
1987-07-01    6.84
1987-10-01    6.92
1988-01-01    6.66
1988-04-01    7.16
1988-07-01    7.98
Name: i, dtype: float64
date
2006-04-01    4.91
2006-07-01    5.25
2006-10-01    5.25
2007-01-01    5.26
2007-04-01    5.25
Name: i, dtype: float64
             pi         y     i
date                           
1987Q3  2.66122 -0.388640  6.84
1987Q4  2.91876  0.526730  6.92
1988Q1  3.06577  0.260731  6.66
1988Q2  3.35274  0.78885

In [53]:
# Set Alias and Check
df_all = df.copy()
print(df_all.head())
print(df_all.tail())

             pi         y     i
date                           
1987Q3  2.66122 -0.388640  6.84
1987Q4  2.91876  0.526730  6.92
1988Q1  3.06577  0.260731  6.66
1988Q2  3.35274  0.788853  7.16
1988Q3  3.80207  0.595530  7.98
             pi         y     i
date                           
2006Q2  3.35642  1.319357  4.91
2006Q3  3.13805  0.978513  5.25
2006Q4  2.66316  1.380249  5.25
2007Q1  2.91019  1.210046  5.26
2007Q2  2.72883  1.336951  5.25


## Model Buildout